# GaussianPro × Scaffold-GS — Validation 90/10

Notebook này train 30.000 iteration với split cố định 90% train / 10% validation. Validation chạy tại 12k và sau đó mỗi 3k iteration, ghi PSNR, SSIM, LPIPS và loss vào CSV. Biểu đồ được làm mới trong khi train.

In [ ]:
from pathlib import Path
import gc, json, os, shutil, subprocess, sys, time, zipfile
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display

# ===== CẤU HÌNH =====
SCENE_NAME = 'HCM0539'  # đổi scene tại đây
DATA_ROOT = Path('/kaggle/input/datasets/acomingzzz/maindataset')
REPO_DIR = Path('/kaggle/working/round2_kusanagi')
OUTPUT_ROOT = Path('/kaggle/working/validation_gaussianpro_90_10')
MODEL_DIR = OUTPUT_ROOT / SCENE_NAME
ITERATIONS = 30_000
EVAL_ITERATIONS = list(range(12_000, ITERATIONS + 1, 3_000))
VALIDATION_RATIO = 0.10
VALIDATION_SEED = 42
MONITOR_SECONDS = 30
GPU = '0'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Evaluation iterations:', EVAL_ITERATIONS)


In [ ]:
# Chuẩn bị source nếu repo chưa tồn tại trong /kaggle/working.
if not (REPO_DIR / 'train.py').exists():
    archives = list(Path('/kaggle/input').rglob('kusanagi-source.zip'))
    if len(archives) != 1:
        raise RuntimeError(f'Cần đúng 1 kusanagi-source.zip, tìm thấy: {archives}')
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(REPO_DIR)

def resolve_scene(data_root: Path, scene_name: str) -> Path:
    candidates = [data_root / scene_name, data_root / scene_name / 'train']
    candidates += list(data_root.rglob(scene_name))
    for candidate in candidates:
        roots = [candidate, candidate / 'train']
        for root in roots:
            if (root / 'sparse').exists() and (root / 'images').exists():
                return root
    raise FileNotFoundError(f'Không tìm thấy scene {scene_name} dưới {data_root}')

SCENE_ROOT = resolve_scene(DATA_ROOT, SCENE_NAME)
print('Repo :', REPO_DIR)
print('Scene:', SCENE_ROOT)
print('Model:', MODEL_DIR)


In [ ]:
TRAIN_CSV = MODEL_DIR / 'train_curve.csv'
METRICS_CSV = MODEL_DIR / 'validation_metrics.csv'
LOG_PATH = MODEL_DIR / 'validation_train.log'

def read_csv_safe(path):
    try:
        return pd.read_csv(path) if path.exists() else pd.DataFrame()
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        return pd.DataFrame()

def plot_monitor(final=False):
    train_df = read_csv_safe(TRAIN_CSV)
    metrics_df = read_csv_safe(METRICS_CSV)
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f'{SCENE_NAME} — GaussianPro 90/10' + (' (final)' if final else ' (running)'), fontsize=16)

    ax = axes[0, 0]
    if not train_df.empty:
        smooth = train_df['train_total_loss'].rolling(10, min_periods=1).mean()
        ax.plot(train_df['iteration'], smooth, label='Train total loss (rolling 10)', color='tab:blue')
    if not metrics_df.empty:
        for split, color in [('train_eval', 'tab:green'), ('val', 'tab:red')]:
            part = metrics_df[metrics_df['split'] == split]
            ax.plot(part['iteration'], part['photo_loss'], 'o-', label=f'{split} photo loss', color=color)
    ax.set_title('Train và validation loss')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Loss'); ax.grid(alpha=.3); ax.legend()

    for ax, metric, title, better in [
        (axes[0, 1], 'psnr', 'PSNR (dB)', 'cao hơn tốt hơn'),
        (axes[1, 0], 'ssim', 'SSIM', 'cao hơn tốt hơn'),
        (axes[1, 1], 'lpips', 'LPIPS', 'thấp hơn tốt hơn'),
    ]:
        if not metrics_df.empty:
            for split, color in [('train_eval', 'tab:green'), ('val', 'tab:red')]:
                part = metrics_df[metrics_df['split'] == split]
                ax.plot(part['iteration'], part[metric], 'o-', label=split, color=color)
        ax.set_title(f'{title} — {better}')
        ax.set_xlabel('Iteration'); ax.set_ylabel(metric.upper()); ax.grid(alpha=.3); ax.legend()
    plt.tight_layout()
    plt.show()

    if not metrics_df.empty:
        display(metrics_df.sort_values(['iteration', 'split']).tail(14))
    if LOG_PATH.exists():
        lines = LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()
        useful = [line for line in lines if 'Propagation' in line or 'Evaluating' in line or 'views):' in line]
        print('\n'.join(useful[-12:]))


In [ ]:
# Train và refresh monitor mỗi 30 giây.
MODEL_DIR.mkdir(parents=True, exist_ok=True)
for stale in (TRAIN_CSV, METRICS_CSV, LOG_PATH):
    if stale.exists():
        stale.unlink()

cmd = [
    sys.executable, 'train.py',
    '-s', str(SCENE_ROOT), '-m', str(MODEL_DIR),
    '-r', '1', '--data_device', 'cpu', '--appearance_dim', '0',
    '--validation_ratio', str(VALIDATION_RATIO),
    '--validation_seed', str(VALIDATION_SEED),
    '--gpu', GPU,
    '--iterations', str(ITERATIONS),
    '--test_iterations', *map(str, EVAL_ITERATIONS),
    '--save_iterations', str(ITERATIONS),
    '--use_gaussianpro',
    '--gaussianpro_start_iter', '1500',
    '--gaussianpro_until_iter', str(ITERATIONS),
    '--gaussianpro_interval', '50',
    '--gaussianpro_neighbors', '4',
    '--gaussianpro_downsample', '4',
    '--gaussianpro_patch_radius', '2',
    '--gaussianpro_patchmatch_iterations', '3',
    '--gaussianpro_min_consistent_views', '2',
    '--gaussianpro_max_photo_error', '0.35',
    '--gaussianpro_max_anchors_per_step', '512',
    '--gaussianpro_voxel_factor', '0.75',
    '--lambda_dssim', '0.2',
]
print(' '.join(cmd))
started = time.time()
with LOG_PATH.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=log_handle, stderr=subprocess.STDOUT, text=True)
    while process.poll() is None:
        time.sleep(MONITOR_SECONDS)
        clear_output(wait=True)
        print(f'Đang train: {(time.time()-started)/60:.1f} phút | PID={process.pid}')
        plot_monitor(final=False)
    return_code = process.wait()

clear_output(wait=True)
plot_monitor(final=True)
print(f'Hoàn tất sau {(time.time()-started)/60:.1f} phút, return_code={return_code}')
if return_code != 0:
    print(LOG_PATH.read_text(encoding='utf-8', errors='replace')[-12000:])
    raise RuntimeError('Training thất bại; xem validation_train.log')


In [ ]:
# Bảng tổng kết validation và checkpoint tốt nhất theo từng metric.
metrics = pd.read_csv(METRICS_CSV)
val = metrics[metrics['split'] == 'val'].sort_values('iteration').copy()
display(val[['iteration', 'count', 'photo_loss', 'psnr', 'ssim', 'lpips', 'anchors']])

summary = pd.DataFrame([
    {'metric': 'photo_loss', 'best_iteration': int(val.loc[val.photo_loss.idxmin(), 'iteration']), 'best_value': val.photo_loss.min(), 'direction': 'min'},
    {'metric': 'PSNR', 'best_iteration': int(val.loc[val.psnr.idxmax(), 'iteration']), 'best_value': val.psnr.max(), 'direction': 'max'},
    {'metric': 'SSIM', 'best_iteration': int(val.loc[val.ssim.idxmax(), 'iteration']), 'best_value': val.ssim.max(), 'direction': 'max'},
    {'metric': 'LPIPS', 'best_iteration': int(val.loc[val.lpips.idxmin(), 'iteration']), 'best_value': val.lpips.min(), 'direction': 'min'},
])
display(summary)
summary.to_csv(MODEL_DIR / 'validation_summary.csv', index=False)
print('Metrics:', METRICS_CSV)
print('Summary:', MODEL_DIR / 'validation_summary.csv')
